# Temporal Weight Comparison: v1 (tau=24d) vs v2 (tau=6d)

Compare `d_empirical` and classification outputs between v1 and v2 high-danger runs.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import json
import glob
import os
from pathlib import Path
import pandas as pd

V1_DIR = Path('../../local/issw/v1_high_danger_output/sarvalanche_runs')
V2_DIR = Path('../../local/issw/high_danger_output/sarvalanche_runs')
V1_PREDICTIONS = Path('../../local/issw/v1_outputs/all_track_predictions.json')
V2_PARTIAL_DIR = Path('../../local/issw/_partial_predictions')

## 1. Temporal weight comparison

Compare `w_temporal` profiles between v1 (tau=24d) and v2 (tau=6d).

In [ ]:
# Load one example to compare w_temporal
example = 'Banner_Summit_2022-12-12.nc'
ds_v1 = xr.open_dataset(V1_DIR / example)
ds_v2 = xr.open_dataset(V2_DIR / example)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ds_v1.time, ds_v1.w_temporal, 'o-', label='v1 (tau=24d)')
ax.plot(ds_v2.time, ds_v2.w_temporal, 's-', label='v2 (tau=6d)')
ax.set_xlabel('Time')
ax.set_ylabel('Temporal Weight')
ax.set_title(f'Temporal Weights: {example}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. d_empirical comparison across all runs

In [ ]:
# Gather d_empirical stats for all common runs
v2_files = sorted(V2_DIR.glob('*.nc'))
rows = []
for v2f in v2_files:
    name = v2f.name
    v1f = V1_DIR / name
    if not v1f.exists():
        continue
    ds1 = xr.open_dataset(v1f)
    ds2 = xr.open_dataset(v2f)
    if 'd_empirical' not in ds1.data_vars or 'd_empirical' not in ds2.data_vars:
        ds1.close(); ds2.close()
        continue
    d1 = ds1['d_empirical'].values
    d2 = ds2['d_empirical'].values
    
    stem = name.replace('.nc', '')
    date = stem[-10:]
    zone = stem[:-11]
    
    rows.append({
        'zone': zone, 'date': date, 'name': stem,
        'v1_mean': np.nanmean(d1), 'v1_median': np.nanmedian(d1),
        'v1_std': np.nanstd(d1), 'v1_max': np.nanmax(d1),
        'v2_mean': np.nanmean(d2), 'v2_median': np.nanmedian(d2),
        'v2_std': np.nanstd(d2), 'v2_max': np.nanmax(d2),
        'v1_detections': int(ds1['detections'].sum().values),
        'v2_detections': int(ds2['detections'].sum().values),
    })
    ds1.close()
    ds2.close()

df = pd.DataFrame(rows)
df['mean_diff'] = df['v2_mean'] - df['v1_mean']
df['detection_diff'] = df['v2_detections'] - df['v1_detections']
print(f'Loaded {len(df)} runs')
df

In [ ]:
# Scatter: v1 vs v2 mean d_empirical
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(df['v1_mean'], df['v2_mean'], c='steelblue', edgecolors='k', s=50)
lims = [min(df['v1_mean'].min(), df['v2_mean'].min()), max(df['v1_mean'].max(), df['v2_mean'].max())]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1')
ax.set_xlabel('v1 mean d_empirical (tau=24d)')
ax.set_ylabel('v2 mean d_empirical (tau=6d)')
ax.set_title('Mean d_empirical: v1 vs v2')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.scatter(df['v1_detections'], df['v2_detections'], c='coral', edgecolors='k', s=50)
lims = [0, max(df['v1_detections'].max(), df['v2_detections'].max())]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1')
ax.set_xlabel('v1 detections (tau=24d)')
ax.set_ylabel('v2 detections (tau=6d)')
ax.set_title('Detection count: v1 vs v2')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Spatial comparison of d_empirical for a few example runs
examples = df.nlargest(3, 'mean_diff')['name'].tolist() + df.nsmallest(3, 'mean_diff')['name'].tolist()

for stem in examples:
    ds1 = xr.open_dataset(V1_DIR / f'{stem}.nc')
    ds2 = xr.open_dataset(V2_DIR / f'{stem}.nc')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    vmin = min(np.nanpercentile(ds1['d_empirical'].values, 2), np.nanpercentile(ds2['d_empirical'].values, 2))
    vmax = max(np.nanpercentile(ds1['d_empirical'].values, 98), np.nanpercentile(ds2['d_empirical'].values, 98))
    
    ds1['d_empirical'].plot(ax=axes[0], vmin=vmin, vmax=vmax, cmap='viridis')
    axes[0].set_title(f'v1 (tau=24d)\n{stem}')
    
    ds2['d_empirical'].plot(ax=axes[1], vmin=vmin, vmax=vmax, cmap='viridis')
    axes[1].set_title(f'v2 (tau=6d)\n{stem}')
    
    diff = ds2['d_empirical'] - ds1['d_empirical']
    dlim = max(abs(np.nanpercentile(diff.values, 2)), abs(np.nanpercentile(diff.values, 98)))
    diff.plot(ax=axes[2], vmin=-dlim, vmax=dlim, cmap='RdBu_r')
    axes[2].set_title(f'v2 - v1\n{stem}')
    
    plt.tight_layout()
    plt.show()
    ds1.close()
    ds2.close()

## 3. d_empirical distribution comparison

In [ ]:
# Aggregate d_empirical histograms across all runs
all_v1 = []
all_v2 = []
for v2f in sorted(V2_DIR.glob('*.nc')):
    v1f = V1_DIR / v2f.name
    if not v1f.exists():
        continue
    ds1 = xr.open_dataset(v1f)
    ds2 = xr.open_dataset(v2f)
    if 'd_empirical' not in ds1.data_vars or 'd_empirical' not in ds2.data_vars:
        ds1.close(); ds2.close()
        continue
    d1 = ds1['d_empirical'].values.ravel()
    d2 = ds2['d_empirical'].values.ravel()
    all_v1.append(d1[np.isfinite(d1)])
    all_v2.append(d2[np.isfinite(d2)])
    ds1.close()
    ds2.close()

all_v1 = np.concatenate(all_v1)
all_v2 = np.concatenate(all_v2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins = np.linspace(min(all_v1.min(), all_v2.min()), np.percentile(np.concatenate([all_v1, all_v2]), 99), 100)
axes[0].hist(all_v1, bins=bins, alpha=0.6, label='v1 (tau=24d)', density=True)
axes[0].hist(all_v2, bins=bins, alpha=0.6, label='v2 (tau=6d)', density=True)
axes[0].set_xlabel('d_empirical')
axes[0].set_ylabel('Density')
axes[0].set_title('d_empirical Distribution (all runs)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CDF
for vals, label in [(all_v1, 'v1 (tau=24d)'), (all_v2, 'v2 (tau=6d)')]:
    sorted_v = np.sort(vals)
    cdf = np.arange(1, len(sorted_v) + 1) / len(sorted_v)
    idx = np.linspace(0, len(sorted_v) - 1, 5000, dtype=int)
    axes[1].plot(sorted_v[idx], cdf[idx], label=label)
axes[1].set_xlabel('d_empirical')
axes[1].set_ylabel('CDF')
axes[1].set_title('d_empirical CDF')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'v1: mean={all_v1.mean():.4f}, std={all_v1.std():.4f}, median={np.median(all_v1):.4f}')
print(f'v2: mean={all_v2.mean():.4f}, std={all_v2.std():.4f}, median={np.median(all_v2):.4f}')

## 4. Track classification comparison: v1 vs v2 (partial)

In [ ]:
# Load v1 predictions
with open(V1_PREDICTIONS) as f:
    v1_preds = json.load(f)

# Load v2 partial predictions and merge
v2_preds = {}
for fp in sorted(V2_PARTIAL_DIR.glob('*.json')):
    with open(fp) as f:
        v2_preds.update(json.load(f))

print(f'v1 tracks: {len(v1_preds)}')
print(f'v2 partial tracks: {len(v2_preds)}')

# Find common track keys
common_keys = sorted(set(v1_preds.keys()) & set(v2_preds.keys()))
print(f'Common tracks: {len(common_keys)}')

In [ ]:
# Build comparison dataframe
pred_rows = []
for k in common_keys:
    parts = k.split('|')
    zone = parts[0]
    date = parts[1]
    track_id = parts[2]
    v1_p = v1_preds[k]['p_debris']
    v2_p = v2_preds[k]['p_debris']
    pred_rows.append({
        'key': k, 'zone': zone, 'date': date, 'track_id': track_id,
        'v1_p_debris': v1_p, 'v2_p_debris': v2_p,
        'v1_class': 'debris' if v1_p >= 0.5 else 'avalanche',
        'v2_class': 'debris' if v2_p >= 0.5 else 'avalanche',
    })

df_pred = pd.DataFrame(pred_rows)
df_pred['p_diff'] = df_pred['v2_p_debris'] - df_pred['v1_p_debris']
df_pred['class_match'] = df_pred['v1_class'] == df_pred['v2_class']

print(f"Classification agreement: {df_pred['class_match'].mean():.1%}")
print(f"\nv1 class counts:\n{df_pred['v1_class'].value_counts()}")
print(f"\nv2 class counts:\n{df_pred['v2_class'].value_counts()}")
print(f"\nDisagreements: {(~df_pred['class_match']).sum()}")

In [ ]:
# Confusion matrix
ct = pd.crosstab(df_pred['v1_class'], df_pred['v2_class'], margins=True)
print('Confusion matrix (v1 rows, v2 cols):')
ct

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# p_debris scatter
ax = axes[0]
ax.scatter(df_pred['v1_p_debris'], df_pred['v2_p_debris'], alpha=0.3, s=10, c='steelblue')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.axhline(0.5, color='red', alpha=0.3, ls=':')
ax.axvline(0.5, color='red', alpha=0.3, ls=':')
ax.set_xlabel('v1 p_debris (tau=24d)')
ax.set_ylabel('v2 p_debris (tau=6d)')
ax.set_title('Track p_debris: v1 vs v2')
ax.grid(True, alpha=0.3)

# p_debris difference histogram
ax = axes[1]
ax.hist(df_pred['p_diff'], bins=50, edgecolor='k', alpha=0.7)
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('v2 - v1 p_debris')
ax.set_ylabel('Count')
ax.set_title('p_debris Difference Distribution')
ax.grid(True, alpha=0.3)

# Agreement by zone/date
ax = axes[2]
agree_by_run = df_pred.groupby(['zone', 'date'])['class_match'].mean().reset_index()
agree_by_run['label'] = agree_by_run['zone'].str[:10] + ' ' + agree_by_run['date']
agree_by_run = agree_by_run.sort_values('class_match')
ax.barh(range(len(agree_by_run)), agree_by_run['class_match'])
ax.set_yticks(range(len(agree_by_run)))
ax.set_yticklabels(agree_by_run['label'], fontsize=7)
ax.set_xlabel('Classification Agreement Rate')
ax.set_title('Agreement by Run')
ax.axvline(1.0, color='green', ls='--', alpha=0.3)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Look at the disagreements more closely
disagree = df_pred[~df_pred['class_match']].copy()
print(f'Total disagreements: {len(disagree)}')
print(f'\nFlipped to debris (v1=avy, v2=debris): {((disagree.v1_class=="avalanche") & (disagree.v2_class=="debris")).sum()}')
print(f'Flipped to avalanche (v1=debris, v2=avy): {((disagree.v1_class=="debris") & (disagree.v2_class=="avalanche")).sum()}')
print(f'\nDisagreements by zone:')
print(disagree.groupby('zone').size())
print(f'\nDisagreements by date:')
print(disagree.groupby('date').size())

In [ ]:
# Summary stats per zone/date for avalanche counts
summary = df_pred.groupby(['zone', 'date']).agg(
    n_tracks=('key', 'size'),
    v1_avy=('v1_class', lambda x: (x == 'avalanche').sum()),
    v2_avy=('v2_class', lambda x: (x == 'avalanche').sum()),
    v1_mean_p=('v1_p_debris', 'mean'),
    v2_mean_p=('v2_p_debris', 'mean'),
    agreement=('class_match', 'mean'),
).reset_index()
summary['avy_diff'] = summary['v2_avy'] - summary['v1_avy']
summary